<a href="https://colab.research.google.com/github/shivanshi-09/weather_modelling/blob/main/Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -qq diffusers accelerate open-clip-torch gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.0 MB/s eta 0:00:00


In [16]:
import numpy as np
import torch
import torchvision
from diffusers import DDIMScheduler, DDPMPipeline
from PIL import Image
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

pipeline_name = "johnowhitaker/sd-class-wikiart-from-bedrooms"
image_pipe = DDPMPipeline.from_pretrained(pipeline_name).to(device)
scheduler = DDIMScheduler.from_pretrained(pipeline_name)
scheduler.set_timesteps(num_inference_steps=80)

Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


In [20]:
def color_loss(images, target_color=(0.1, 0.9, 0.5)):
    target = torch.tensor(target_color).to(images.device) * 2 - 1
    target = target[None, :, None, None]
    return torch.abs(images - target).mean()

In [21]:
import gradio as gr
import re
from PIL import ImageColor
def generate(color, guidance_loss_scale):
    rgba_match = re.match(r'rgba?\((\d+\.?\d*),\s*(\d+\.?\d*),\s*(\d+\.?\d*)', color)
    if rgba_match:
        r, g, b = [int(float(v)) for v in rgba_match.groups()]
        target_color = [r / 255, g / 255, b / 255]
    else:
        target_color = [a / 255 for a in ImageColor.getcolor(color, "RGB")]

    x = torch.randn(1, 3, 256, 256).to(device)
    for i, t in tqdm(enumerate(scheduler.timesteps)):
        model_input = scheduler.scale_model_input(x, t)
        with torch.no_grad():
            noise_pred = image_pipe.unet(model_input, t)["sample"]
        if i % 2 == 0:
            x = x.detach().requires_grad_()
            x0 = scheduler.step(noise_pred, t, x).pred_original_sample
            loss = color_loss(x0, target_color) * guidance_loss_scale
            cond_grad = -torch.autograd.grad(loss, x)[0]
            x = x.detach() + cond_grad

        x = scheduler.step(noise_pred, t, x).prev_sample

    grid = torchvision.utils.make_grid(x, nrow=1)
    im = grid.permute(1, 2, 0).cpu().clip(-1, 1) * 0.5 + 0.5
    return Image.fromarray(np.array(im * 255).astype(np.uint8))

demo = gr.Interface(
    fn=generate,
    inputs=[
        gr.ColorPicker(label="Target color", value="#55FFAA"),
        gr.Slider(label="Guidance scale", minimum=0, maximum=30, value=8),
    ],
    outputs=gr.Image(label="Result"),
    examples=[["#FF4444", 10], ["#4444FF", 5]],
)
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2539cc78aedc2883f5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


0it [00:00, ?it/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2539cc78aedc2883f5.gradio.live
